In [1]:
import os
from google.colab import drive

print("🔌 正在掛載 Google Drive...")
drive.mount('/content/drive')

print("\n🚀 啟動 A100 極速武裝程序...")

# 1. 安裝圖神經網路核心套件 (PyTorch Geometric)
print("🕸️ 安裝 torch-geometric...")
!pip install -q torch-geometric

# 2. 尋找並安裝你編譯好的 Mamba 專武 (.whl)
# 假設你的 wheel 檔存在這個資料夾，如果路徑不同請幫我微調一下
WHEEL_DIR = '/content/drive/MyDrive/Mamba_Wheels_A100'

print(f"📦 從 {WHEEL_DIR} 載入預編譯套件...")
!pip install -q {WHEEL_DIR}/causal_conv1d-*.whl
!pip install -q {WHEEL_DIR}/mamba_ssm-*.whl

print("\n==========================================")
print("✅ Cell 1 執行完畢！")
print("A100 環境已完美配置 Mamba 與 GNN 套件！")
print("==========================================")

🔌 正在掛載 Google Drive...
Mounted at /content/drive

🚀 啟動 A100 極速武裝程序...
🕸️ 安裝 torch-geometric...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 48.9 MB/s eta 0:00:00
📦 從 /content/drive/MyDrive/Mamba_Wheels_A100 載入預編譯套件...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 10.6 MB/s eta 0:00:00

✅ Cell 1 執行完畢！
A100 環境已完美配置 Mamba 與 GNN 套件！


In [4]:
# ==========================================
# 🧠 V4.0 Mamba 訓練檔 Cell 1: 上帝視角 120D 軌跡資料派發中心 (無警告 60D 巨獸版)
# ==========================================
import os
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from collections import defaultdict
from tqdm.auto import tqdm

from google.colab import drive
drive.mount('/content/drive')

print("🚀 啟動 V4.0 上帝視角資料鍛造廠 (60D 歷史記憶版)...")

ROOT_DIR = '/content/drive/MyDrive/MarketMamba_V4'
data_path = os.path.join(ROOT_DIR, 'Processed_Features', 'Mamba_60D_Matrix.parquet')

df = pd.read_parquet(data_path)
df = df.sort_values(['stock_id', 'Date']).reset_index(drop=True)

exclude_cols = ['stock_id', 'Date', 'Future_5d_Return']
feature_cols = [col for col in df.columns if col not in exclude_cols]

# === 動態生成 30 天未來軌跡 ===
PRED_DAYS = 30
full_Y = np.zeros((len(df), PRED_DAYS), dtype=np.float32)
g_close = df.groupby('stock_id')['Close']
current_close = df['Close'].values

for i in tqdm(range(1, PRED_DAYS + 1), desc="計算未來 30D 軌跡"):
    shifted_close = g_close.shift(-i).values
    returns = np.where(pd.isna(shifted_close), 0.0, (shifted_close - current_close) / (current_close + 1e-8))
    full_Y[:, i-1] = returns

# === 特徵標準化 ===
train_mask = df['Date'] < '2025-01-01'
scaler = StandardScaler()
scaler.fit(df.loc[train_mask, feature_cols])

full_X = scaler.transform(df[feature_cols]).astype(np.float32)
dates = df['Date'].values
stock_ids = df['stock_id'].values

# === 建立上帝視角索引 (120天防漏電) ===
# ⚠️ 升級點：確保每筆資料往前算有 120 天
changes = np.where(stock_ids[:-1] != stock_ids[1:])[0] + 1
start_idx = 0
valid_mask = np.zeros(len(df), dtype=bool)

for end_idx in np.append(changes, len(df)):
    if end_idx - start_idx >= 120:  # ⚠️ 這裡改成 60
        valid_mask[start_idx+119 : end_idx] = True
    start_idx = end_idx

date_to_indices = defaultdict(list)
valid_row_indices = np.where(valid_mask)[0]

for idx in valid_row_indices:
    date_to_indices[dates[idx]].append(idx)

unique_valid_dates = np.array(sorted(list(date_to_indices.keys())))
train_dates = unique_valid_dates[unique_valid_dates < np.datetime64('2025-01-01')]
val_dates = unique_valid_dates[(unique_valid_dates >= np.datetime64('2025-01-01')) & (unique_valid_dates < np.datetime64('2026-01-01'))]
test_dates = unique_valid_dates[unique_valid_dates >= np.datetime64('2026-01-01')]

class GodModeCrossSectionalDataset(Dataset):
    def __init__(self, full_X, full_Y, date_to_indices, active_dates, seq_len=60):
        self.full_X = full_X
        self.full_Y = full_Y
        self.date_to_indices = date_to_indices
        self.active_dates = active_dates
        self.seq_len = seq_len

    def __len__(self):
        return len(self.active_dates)

    def __getitem__(self, idx):
        target_date = self.active_dates[idx]
        row_indices = self.date_to_indices[target_date]
        num_active_stocks = len(row_indices)

        X_batch = np.zeros((num_active_stocks, self.seq_len, self.full_X.shape[1]), dtype=np.float32)
        Y_batch = np.zeros((num_active_stocks, self.full_Y.shape[1]), dtype=np.float32)

        for i, row_idx in enumerate(row_indices):
            X_batch[i] = self.full_X[row_idx - self.seq_len + 1 : row_idx + 1]
            Y_batch[i] = self.full_Y[row_idx]

        return torch.tensor(X_batch), torch.tensor(Y_batch)

# 🚨 終極修復點：num_workers=0 拔除多執行緒報錯
train_dataset = GodModeCrossSectionalDataset(full_X, full_Y, date_to_indices, train_dates, seq_len=120)
val_dataset = GodModeCrossSectionalDataset(full_X, full_Y, date_to_indices, val_dates, seq_len=120)
test_dataset = GodModeCrossSectionalDataset(full_X, full_Y, date_to_indices, test_dates, seq_len=120)

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=0, pin_memory=True)

print("✅ Route B 巨獸級 DataLoader 準備完畢！")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🚀 啟動 V4.0 上帝視角資料鍛造廠 (60D 歷史記憶版)...


計算未來 30D 軌跡:   0%|          | 0/30 [00:00<?, ?it/s]

✅ Route B 巨獸級 DataLoader 準備完畢！


下面這段是給最後一波訓練得時候用的

In [5]:
# ==========================================
# 🧠 V4.0 Mamba 訓練檔 Cell 1: 最後上線前的那一次煉丹要用的，全資料訓練版
# ==========================================
import os
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from collections import defaultdict
from tqdm.auto import tqdm

from google.colab import drive
drive.mount('/content/drive')

# 👇 補上這段 Dataset 藍圖，讓 Cell 3 可以獨立運作
class GodModeCrossSectionalDataset(Dataset):
    def __init__(self, X, Y, date_to_indices, dates, seq_len=120):
        self.X = X
        self.Y = Y
        self.date_to_indices = date_to_indices
        self.dates = dates
        self.seq_len = seq_len

    def __len__(self):
        return len(self.dates)

    def __getitem__(self, idx):
        date = self.dates[idx]
        indices = self.date_to_indices[date]
        # X: [Num_Stocks, Seq_Len, Features], Y: [Num_Stocks, Pred_Days]
        return self.X[indices], self.Y[indices]

# 👇 原本的這幾行保持不變
test_dates = []  # 實盤重訓不需要 Test
train_dataset = GodModeCrossSectionalDataset(full_X, full_Y, date_to_indices, train_dates, seq_len=120)
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, num_workers=0, pin_memory=True)

print(f"✅ 全資料 DataLoader 準備完畢！總天數: {len(train_loader)}")

print("🚀 啟動 V4.0 上帝視角資料鍛造廠 (60D 歷史記憶版)...")

ROOT_DIR = '/content/drive/MyDrive/MarketMamba_V4'
data_path = os.path.join(ROOT_DIR, 'Processed_Features', 'Mamba_60D_Matrix.parquet')

df = pd.read_parquet(data_path)
df = df.sort_values(['stock_id', 'Date']).reset_index(drop=True)

exclude_cols = ['stock_id', 'Date', 'Future_5d_Return']
feature_cols = [col for col in df.columns if col not in exclude_cols]

# === 動態生成 30 天未來軌跡 ===
PRED_DAYS = 30
full_Y = np.zeros((len(df), PRED_DAYS), dtype=np.float32)
g_close = df.groupby('stock_id')['Close']
current_close = df['Close'].values

for i in tqdm(range(1, PRED_DAYS + 1), desc="計算未來 30D 軌跡"):
    shifted_close = g_close.shift(-i).values
    returns = np.where(pd.isna(shifted_close), 0.0, (shifted_close - current_close) / (current_close + 1e-8))
    full_Y[:, i-1] = returns

# === 特徵標準化 ===
train_mask = df['Date'] < '2025-01-01'
scaler = StandardScaler()
scaler.fit(df.loc[train_mask, feature_cols])

full_X = scaler.transform(df[feature_cols]).astype(np.float32)
dates = df['Date'].values
stock_ids = df['stock_id'].values

# === 建立上帝視角索引 (120天防漏電) ===
# ⚠️ 升級點：確保每筆資料往前算有 120 天
changes = np.where(stock_ids[:-1] != stock_ids[1:])[0] + 1
start_idx = 0
valid_mask = np.zeros(len(df), dtype=bool)

for end_idx in np.append(changes, len(df)):
    if end_idx - start_idx >= 120:  # ⚠️ 這裡改成 60
        valid_mask[start_idx+119 : end_idx] = True
    start_idx = end_idx

date_to_indices = defaultdict(list)
valid_row_indices = np.where(valid_mask)[0]

for idx in valid_row_indices:
    date_to_indices[dates[idx]].append(idx)

unique_valid_dates = np.array(sorted(list(date_to_indices.keys())))

train_dates = unique_valid_dates # 💯 把 2019 到今天的所有資料都餵給模型
val_dates = []   # 實盤重訓不需要 Val
test_dates = []  # 實盤重訓不需要 Test

train_dataset = GodModeCrossSectionalDataset(full_X, full_Y, date_to_indices, train_dates, seq_len=120)
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, num_workers=0, pin_memory=True)

class GodModeCrossSectionalDataset(Dataset):
    def __init__(self, full_X, full_Y, date_to_indices, active_dates, seq_len=60):
        self.full_X = full_X
        self.full_Y = full_Y
        self.date_to_indices = date_to_indices
        self.active_dates = active_dates
        self.seq_len = seq_len

    def __len__(self):
        return len(self.active_dates)

    def __getitem__(self, idx):
        target_date = self.active_dates[idx]
        row_indices = self.date_to_indices[target_date]
        num_active_stocks = len(row_indices)

        X_batch = np.zeros((num_active_stocks, self.seq_len, self.full_X.shape[1]), dtype=np.float32)
        Y_batch = np.zeros((num_active_stocks, self.full_Y.shape[1]), dtype=np.float32)

        for i, row_idx in enumerate(row_indices):
            X_batch[i] = self.full_X[row_idx - self.seq_len + 1 : row_idx + 1]
            Y_batch[i] = self.full_Y[row_idx]

        return torch.tensor(X_batch), torch.tensor(Y_batch)

# 🚨 終極修復點：num_workers=0 拔除多執行緒報錯
train_dataset = GodModeCrossSectionalDataset(full_X, full_Y, date_to_indices, train_dates, seq_len=120)
val_dataset = GodModeCrossSectionalDataset(full_X, full_Y, date_to_indices, val_dates, seq_len=120)
test_dataset = GodModeCrossSectionalDataset(full_X, full_Y, date_to_indices, test_dates, seq_len=120)

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=0, pin_memory=True)

print("✅ Route B 巨獸級 DataLoader 準備完畢！")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 全資料 DataLoader 準備完畢！總天數: 1259
🚀 啟動 V4.0 上帝視角資料鍛造廠 (60D 歷史記憶版)...


計算未來 30D 軌跡:   0%|          | 0/30 [00:00<?, ?it/s]

✅ Route B 巨獸級 DataLoader 準備完畢！


上面這段是給最後一波訓練得時候用的

In [ ]:
# ==========================================
# 🧠 V4.0 Mamba 訓練檔 Cell 2: 上帝視角 Mamba (A100 解放版)
# ==========================================
import torch
import torch.nn as nn
import math

try:
    from mamba_ssm import Mamba
    print("✅ Mamba-SSM 套件載入成功！")
except ImportError:
    raise ImportError("❌ 找不到 mamba-ssm！請先執行 !pip install mamba-ssm causal-conv1d")

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class MarketMambaV4_GodMode(nn.Module):
    # 🚨 升級點：把 d_state 加入參數列，讓它可以被我們從外部放大！
    def __init__(self, input_dim=56, seq_len=120, d_model=256, n_heads=16, pred_days=30, num_mamba_layers=4, d_state=32, dropout_rate=0.3):
        super().__init__()

        print(f"🌍 啟動 V4.0 上帝視角架構 (d_model={d_model}, layers={num_mamba_layers}, d_state={d_state})...")
        self.pred_days = pred_days

        self.embedding = nn.Sequential(
            nn.Linear(input_dim, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout_rate)
        )
        self.pos_encoder = PositionalEncoding(d_model=d_model, max_len=seq_len)

        self.mamba_layers = nn.ModuleList()
        self.mamba_norms = nn.ModuleList()
        for _ in range(num_mamba_layers):
            # 🚨 升級點：不再寫死 32，改吃外部傳進來的 d_state
            self.mamba_layers.append(Mamba(d_model=d_model, d_state=d_state, d_conv=4, expand=2))
            self.mamba_norms.append(nn.LayerNorm(d_model))

        self.dropout = nn.Dropout(dropout_rate)

        self.attn_norm = nn.LayerNorm(d_model)
        self.cross_stock_attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=n_heads,
            dropout=dropout_rate,
            batch_first=True
        )

        self.trajectory_head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(d_model, pred_days)
        )

    def forward(self, x):
        x = self.embedding(x)
        x = self.pos_encoder(x)

        for mamba, norm in zip(self.mamba_layers, self.mamba_norms):
            x = x + self.dropout(mamba(norm(x)))

        x_temporal = x[:, -1, :]

        x_graph = x_temporal.unsqueeze(0)
        norm_x = self.attn_norm(x_graph)

        attn_out, _ = self.cross_stock_attn(norm_x, norm_x, norm_x)
        x_graph = x_graph + self.dropout(attn_out)

        x_final = x_graph.squeeze(0)
        out_trajectory = self.trajectory_head(x_final)

        return out_trajectory

✅ Mamba-SSM 套件載入成功！


In [ ]:
# ==========================================
# 🧠 V4.0 Mamba 訓練檔 Cell 3: 機構級量化煉丹爐 (支援斷點接力訓練版)
# ==========================================
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm.auto import tqdm
import os

print("🔥 點火！啟動 V4.0 機構級煉丹爐 (支援跨電腦接力)...")

# ...(QuantTrajectoryLoss 定義與前面相同，省略保持版面簡潔)...
class QuantTrajectoryLoss(nn.Module):
    def __init__(self, pred_days=30, decay_rate=0.92, directional_penalty=2.5):
        super().__init__()
        self.pred_days = pred_days
        self.penalty = directional_penalty
        self.huber = nn.SmoothL1Loss(reduction='none')
        weights = torch.tensor([decay_rate ** i for i in range(pred_days)], dtype=torch.float32)
        self.register_buffer('time_weights', (weights / weights.sum()) * pred_days)

    def forward(self, pred, target):
        base_loss = self.huber(pred, target)
        sign_match = pred * target
        dir_penalty_mask = torch.where(sign_match < 0, self.penalty, 1.0)
        valid_mask = torch.where(target == 0.0, 0.0, 1.0)
        final_loss = base_loss * self.time_weights.unsqueeze(0) * dir_penalty_mask * valid_mask
        return final_loss.sum() / (valid_mask.sum() + 1e-8)

# === 超參數與環境 ===
EPOCHS = 50
LEARNING_RATE = 1.5e-4
WEIGHT_DECAY = 5e-5
ACCUMULATION_STEPS = 16
PATIENCE = 6

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = MarketMambaV4_GodMode(
    input_dim=len(feature_cols), seq_len=120, d_model=512,
    n_heads=32, pred_days=30, num_mamba_layers=8, d_state=64, dropout_rate=0.4
).to(device)

criterion = QuantTrajectoryLoss().to(device)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler = torch.amp.GradScaler('cuda')

save_dir = '/content/drive/MyDrive/MarketMamba_V4/Models'
os.makedirs(save_dir, exist_ok=True)
best_model_path = os.path.join(save_dir, 'V4_GodMode_Leviathan_Best.pth')
resume_checkpoint_path = os.path.join(save_dir, 'V4_GodMode_Resume.pth') # 📌 接力存檔點

# ==========================================
# 🔄 斷點恢復機制 (Checkpoint Loading)
# ==========================================
start_epoch = 0
best_val_loss = float('inf')
patience_counter = 0

if os.path.exists(resume_checkpoint_path):
    print(f"🔄 偵測到中斷的訓練進度！正在從雲端硬碟恢復大腦記憶...")
    checkpoint = torch.load(resume_checkpoint_path)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    scaler.load_state_dict(checkpoint['scaler_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    best_val_loss = checkpoint['best_val_loss']
    patience_counter = checkpoint['patience_counter']
    print(f"✅ 成功恢復！將從 Epoch {start_epoch+1} 繼續為您煉丹。")

# === 主訓練迴圈 ===
for epoch in range(start_epoch, EPOCHS):
    model.train()
    train_loss = 0.0
    optimizer.zero_grad()

    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1:02d}/{EPOCHS} [Train]", leave=False)
    for i, (X_batch, Y_batch) in enumerate(train_pbar):
        X, Y = X_batch.squeeze(0).to(device), Y_batch.squeeze(0).to(device)

        with torch.autocast(device_type='cuda', dtype=torch.float16):
            loss = criterion(model(X), Y) / ACCUMULATION_STEPS

        scaler.scale(loss).backward()

        if (i + 1) % ACCUMULATION_STEPS == 0 or (i + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        train_loss += loss.item() * ACCUMULATION_STEPS
        train_pbar.set_postfix({'loss': f"{loss.item() * ACCUMULATION_STEPS:.4f}"})

    avg_train_loss = train_loss / len(train_loader)

    # === 驗證階段 ===
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_batch, Y_batch in tqdm(val_loader, desc=f"Epoch {epoch+1:02d}/{EPOCHS} [Valid]", leave=False):
            X, Y = X_batch.squeeze(0).to(device), Y_batch.squeeze(0).to(device)
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                val_loss += criterion(model(X), Y).item()

    avg_val_loss = val_loss / len(val_loader)
    scheduler.step()

    print(f"📈 Epoch {epoch+1:02d}/{EPOCHS} | Train Loss: {avg_train_loss:.5f} | Val Loss: {avg_val_loss:.5f} | LR: {scheduler.get_last_lr()[0]:.2e}")

    # 📌 每次 Epoch 結束，強制存下接力檔
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'best_val_loss': best_val_loss,
        'patience_counter': patience_counter
    }, resume_checkpoint_path)

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), best_model_path)
        print(f"   🏆 創下新低 Val Loss! 利維坦級模型已存檔: {best_model_path}")
    else:
        patience_counter += 1
        print(f"   ⚠️ Val Loss 停滯，耐心值: {patience_counter}/{PATIENCE}")

    if patience_counter >= PATIENCE:
        print(f"\n🛑 早停機制啟動！巨獸已在 Epoch {epoch+1} 完成進化。")
        break

print("\n🎉 V4.0 巨獸級神經網路煉丹完成！")

🔥 點火！啟動 V4.0 機構級煉丹爐 (支援跨電腦接力)...
🌍 啟動 V4.0 上帝視角架構 (d_model=512, layers=8, d_state=64)...
🔄 偵測到中斷的訓練進度！正在從雲端硬碟恢復大腦記憶...
✅ 成功恢復！將從 Epoch 10 繼續為您煉丹。


Epoch 10/50 [Train]:   0%|          | 0/1259 [00:00<?, ?it/s]

Epoch 10/50 [Valid]:   0%|          | 0/243 [00:00<?, ?it/s]

📈 Epoch 10/50 | Train Loss: 0.00718 | Val Loss: 0.00723 | LR: 1.36e-04
   ⚠️ Val Loss 停滯，耐心值: 1/6


Epoch 11/50 [Train]:   0%|          | 0/1259 [00:00<?, ?it/s]

...

Epoch 17/50 [Valid]:   0%|          | 0/243 [00:00<?, ?it/s]

📈 Epoch 17/50 | Train Loss: 0.00724 | Val Loss: 0.00660 | LR: 1.11e-04
   ⚠️ Val Loss 停滯，耐心值: 5/6


Epoch 18/50 [Train]:   0%|          | 0/1259 [00:00<?, ?it/s]

Epoch 18/50 [Valid]:   0%|          | 0/243 [00:00<?, ?it/s]

📈 Epoch 18/50 | Train Loss: 0.00718 | Val Loss: 0.00679 | LR: 1.07e-04
   ⚠️ Val Loss 停滯，耐心值: 6/6

🛑 早停機制啟動！巨獸已在 Epoch 18 完成進化。

🎉 V4.0 巨獸級神經網路煉丹完成！


In [ ]:
# ==========================================
# 🛑 MarketMamba V3.1: 自動關機防燒錢程序 (Cell 5)
# ==========================================
from google.colab import runtime

print("💤 今日推論與推播已全數完成。")
print("🔌 為了節省算力點數，系統將在 3 秒後自動切斷執行階段電源...")
import time
time.sleep(3)

# 執行自爆指令，強制回收虛擬機與 GPU
runtime.unassign()

💤 今日推論與推播已全數完成。
🔌 為了節省算力點數，系統將在 3 秒後自動切斷執行階段電源...


In [ ]:
# ==========================================
# 🏥 V4.0 巨獸腦部微創手術 (Transfer Learning 快速修復 - 完整獨立版)
# ==========================================
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm.auto import tqdm
import os

print("🏥 準備進行腦部微創手術，切除坍塌的 Attention 層...")

# 🚨 補上遺漏的 Loss 函數定義，讓腳本可以 100% 獨立運行
class QuantTrajectoryLoss(nn.Module):
    def __init__(self, pred_days=30, decay_rate=0.92, directional_penalty=2.5):
        super().__init__()
        self.pred_days = pred_days
        self.penalty = directional_penalty
        self.huber = nn.SmoothL1Loss(reduction='none')
        weights = torch.tensor([decay_rate ** i for i in range(pred_days)], dtype=torch.float32)
        self.register_buffer('time_weights', (weights / weights.sum()) * pred_days)

    def forward(self, pred, target):
        base_loss = self.huber(pred, target)
        sign_match = pred * target
        dir_penalty_mask = torch.where(sign_match < 0, self.penalty, 1.0)
        valid_mask = torch.where(target == 0.0, 0.0, 1.0)
        final_loss = base_loss * self.time_weights.unsqueeze(0) * dir_penalty_mask * valid_mask
        return final_loss.sum() / (valid_mask.sum() + 1e-8)

# 1. 定義切除 Attention 後的手術版架構
class MarketMambaV4_Surgery(nn.Module):
    def __init__(self, input_dim=len(feature_cols), seq_len=120, d_model=512, pred_days=30, num_mamba_layers=8, d_state=64, dropout_rate=0.4):
        super().__init__()
        self.embedding = nn.Sequential(
            nn.Linear(input_dim, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout_rate)
        )
        self.pos_encoder = PositionalEncoding(d_model=d_model, max_len=seq_len)

        self.mamba_layers = nn.ModuleList()
        self.mamba_norms = nn.ModuleList()
        for _ in range(num_mamba_layers):
            self.mamba_layers.append(Mamba(d_model=d_model, d_state=d_state, d_conv=4, expand=2))
            self.mamba_norms.append(nn.LayerNorm(d_model))

        self.dropout = nn.Dropout(dropout_rate)

        # 🌟 全新的預測輸出頭
        self.trajectory_head = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(d_model // 2, pred_days)
        )

    def forward(self, x):
        x = self.embedding(x)
        x = self.pos_encoder(x)
        for mamba, norm in zip(self.mamba_layers, self.mamba_norms):
            x = x + self.dropout(mamba(norm(x)))

        x_temporal = x[:, -1, :] # 提取出完美健康的 Mamba 特徵

        # 🔪 繞過 Attention，直接接入新頭！
        return self.trajectory_head(x_temporal)

# 2. 載入並轉移那顆花 24 小時練出的黃金大腦
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("🧠 正在載入 A100 黃金大腦權重...")

old_model_path = '/content/drive/MyDrive/MarketMamba_V4/Models/V4_GodMode_Leviathan_Best.pth'
old_model = MarketMambaV4_GodMode(
    input_dim=len(feature_cols), seq_len=120, d_model=512,
    n_heads=32, pred_days=30, num_mamba_layers=8, d_state=64, dropout_rate=0.4
).to(device)
old_model.load_state_dict(torch.load(old_model_path, map_location=device))

surgery_model = MarketMambaV4_Surgery().to(device)

# 將健康的器官移植過去
surgery_model.embedding.load_state_dict(old_model.embedding.state_dict())
surgery_model.mamba_layers.load_state_dict(old_model.mamba_layers.state_dict())
surgery_model.mamba_norms.load_state_dict(old_model.mamba_norms.state_dict())
print("✅ 權重移植成功！")

# 3. ❄️ 凍結 Mamba，只訓練新裝上的輸出頭
for param in surgery_model.embedding.parameters(): param.requires_grad = False
for param in surgery_model.mamba_layers.parameters(): param.requires_grad = False
for param in surgery_model.mamba_norms.parameters(): param.requires_grad = False

# 檢查一下是不是只剩下 head 要訓練
trainable_params = sum(p.numel() for p in surgery_model.parameters() if p.requires_grad)
print(f"⚡ 凍結完成！現在只需訓練 {trainable_params:,} 個參數 (原本是千萬級別！)")

# 4. 快速微調訓練迴圈 (大概只要跑 10 個 Epoch 以內)
SURGERY_EPOCHS = 10
optimizer = optim.AdamW(surgery_model.trajectory_head.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = QuantTrajectoryLoss().to(device)

surgery_save_path = '/content/drive/MyDrive/MarketMamba_V4/Models/V4_Surgery_Best.pth'
best_val_loss = float('inf')

print("🚀 啟動光速微調...")
for epoch in range(SURGERY_EPOCHS):
    surgery_model.train()
    train_loss = 0.0

    train_pbar = tqdm(train_loader, desc=f"Surgery Epoch {epoch+1:02d}/{SURGERY_EPOCHS} [Train]", leave=False)
    for X_batch, Y_batch in train_pbar:
        X, Y = X_batch.squeeze(0).to(device), Y_batch.squeeze(0).to(device)

        optimizer.zero_grad()
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            preds = surgery_model(X)
            loss = criterion(preds, Y)

        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    # 驗證
    surgery_model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_batch, Y_batch in tqdm(val_loader, desc=f"[Valid]", leave=False):
            X, Y = X_batch.squeeze(0).to(device), Y_batch.squeeze(0).to(device)
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                val_loss += criterion(surgery_model(X), Y).item()

    avg_val_loss = val_loss / len(val_loader)

    print(f"📈 Epoch {epoch+1:02d} | Train Loss: {avg_train_loss:.5f} | Val Loss: {avg_val_loss:.5f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(surgery_model.state_dict(), surgery_save_path)
        print(f"   🏆 手術模型已存檔: {surgery_save_path}")

print("🎉 腦部手術大成功！現在去推論腳本換上新模型吧！")

🏥 準備進行腦部微創手術，切除坍塌的 Attention 層...
🧠 正在載入 A100 黃金大腦權重...
🌍 啟動 V4.0 上帝視角架構 (d_model=512, layers=8, d_state=64)...
✅ 權重移植成功！
⚡ 凍結完成！現在只需訓練 139,038 個參數 (原本是千萬級別！)
🚀 啟動光速微調...


Surgery Epoch 01/10 [Train]:   0%|          | 0/1259 [00:00<?, ?it/s]

[Valid]:   0%|          | 0/243 [00:00<?, ?it/s]

📈 Epoch 01 | Train Loss: 0.01530 | Val Loss: 0.00805
   🏆 手術模型已存檔: /content/drive/MyDrive/MarketMamba_V4/Models/V4_Surgery_Best.pth


Surgery Epoch 02/10 [Train]:   0%|          | 0/1259 [00:00<?, ?it/s]

...

[Valid]:   0%|          | 0/243 [00:00<?, ?it/s]

📈 Epoch 09 | Train Loss: 0.00725 | Val Loss: 0.00785


Surgery Epoch 10/10 [Train]:   0%|          | 0/1259 [00:00<?, ?it/s]

[Valid]:   0%|          | 0/243 [00:00<?, ?it/s]

📈 Epoch 10 | Train Loss: 0.00725 | Val Loss: 0.00804
🎉 腦部手術大成功！現在去推論腳本換上新模型吧！


In [ ]:
# ==========================================
# 🏆 最終實盤微創手術 (全數據版 - 差異化學習率)
# ==========================================
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm.auto import tqdm
import os

print("🏆 啟動實盤前最終煉丹 (吸收 2025-2026 最新記憶)...")

# 這裡填入你在階段一觀察到的最佳 Epoch 數 (因為全資料沒有 Validation，我們用盲跑)
FINAL_EPOCHS = 5
ACCUMULATION_STEPS = 16

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 1. 載入我們剛剛在「階段一」微創手術成功的新大腦
best_surgery_path = '/content/drive/MyDrive/MarketMamba_V4/Models/V4_Surgery_Best.pth'
final_model_path = '/content/drive/MyDrive/MarketMamba_V4/Models/V4_GodMode_Production.pth'

final_surgery_model = MarketMambaV4_Surgery(
    input_dim=len(feature_cols), seq_len=120, d_model=512,
    pred_days=30, num_mamba_layers=8, d_state=64, dropout_rate=0.4
).to(device)

final_surgery_model.load_state_dict(torch.load(best_surgery_path, map_location=device))
print("✅ 成功載入已修復的 Mamba 權重！")

# 2. 🛡️ 解凍所有參數，但設定「差異化學習率」
for param in final_surgery_model.parameters():
    param.requires_grad = True

# 讓 Mamba 用龜速微調 (1e-5)，讓 Head 用正常速度適應 (1e-3)
optimizer = optim.AdamW([
    {'params': final_surgery_model.embedding.parameters(), 'lr': 1e-5},
    {'params': final_surgery_model.pos_encoder.parameters(), 'lr': 1e-5},
    {'params': final_surgery_model.mamba_layers.parameters(), 'lr': 1e-5},
    {'params': final_surgery_model.mamba_norms.parameters(), 'lr': 1e-5},
    {'params': final_surgery_model.trajectory_head.parameters(), 'lr': 1e-3}
], weight_decay=1e-4)

criterion = QuantTrajectoryLoss().to(device)

# 3. 執行全數據訓練 (無 Validation 盲跑)
print(f"🚀 啟動全數據煉丹，共計 {FINAL_EPOCHS} 圈...")
for epoch in range(FINAL_EPOCHS):
    final_surgery_model.train()
    train_loss = 0.0
    optimizer.zero_grad()

    train_pbar = tqdm(train_loader, desc=f"Final Epoch {epoch+1:02d}/{FINAL_EPOCHS} [Train]", leave=False)
    for i, (X_batch, Y_batch) in enumerate(train_pbar):
        X, Y = X_batch.squeeze(0).to(device), Y_batch.squeeze(0).to(device)

        with torch.autocast(device_type='cuda', dtype=torch.float16):
            preds = final_surgery_model(X)
            loss = criterion(preds, Y)
            loss = loss / ACCUMULATION_STEPS

        loss.backward()

        if (i + 1) % ACCUMULATION_STEPS == 0 or (i + 1) == len(train_loader):
            # 加上梯度裁剪，防止 Mamba 在接觸新資料時梯度爆炸
            torch.nn.utils.clip_grad_norm_(final_surgery_model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()

        train_loss += loss.item() * ACCUMULATION_STEPS

    avg_train_loss = train_loss / len(train_loader)
    print(f"📈 Final Epoch {epoch+1:02d} | Train Loss: {avg_train_loss:.5f}")

# 存下最終上線權重
torch.save(final_surgery_model.state_dict(), final_model_path)
print(f"🎉 實盤大腦進化完成！已存檔至: {final_model_path}")

🏆 啟動實盤前最終煉丹 (吸收 2025-2026 最新記憶)...


NameError: name 'MarketMambaV4_Surgery' is not defined

In [ ]:
# ==========================================
# 🏆 最終實盤微創手術 (全數據版 - 100% 獨立防呆版)
# ==========================================
import torch
import torch.nn as nn
import torch.optim as optim
import math
from tqdm.auto import tqdm
import os
from mamba_ssm import Mamba

print("🏆 啟動實盤前最終煉丹 (吸收 2025-2026 最新記憶)...")

# ==========================================
# 🧩 1. 補齊所有遺失的神經網路藍圖 (防失憶)
# ==========================================
class QuantTrajectoryLoss(nn.Module):
    def __init__(self, pred_days=30, decay_rate=0.92, directional_penalty=2.5):
        super().__init__()
        self.pred_days = pred_days
        self.penalty = directional_penalty
        self.huber = nn.SmoothL1Loss(reduction='none')
        weights = torch.tensor([decay_rate ** i for i in range(pred_days)], dtype=torch.float32)
        self.register_buffer('time_weights', (weights / weights.sum()) * pred_days)

    def forward(self, pred, target):
        base_loss = self.huber(pred, target)
        sign_match = pred * target
        dir_penalty_mask = torch.where(sign_match < 0, self.penalty, 1.0)
        valid_mask = torch.where(target == 0.0, 0.0, 1.0)
        final_loss = base_loss * self.time_weights.unsqueeze(0) * dir_penalty_mask * valid_mask
        return final_loss.sum() / (valid_mask.sum() + 1e-8)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class MarketMambaV4_Surgery(nn.Module):
    def __init__(self, input_dim, seq_len=120, d_model=512, pred_days=30, num_mamba_layers=8, d_state=64, dropout_rate=0.4):
        super().__init__()
        self.embedding = nn.Sequential(
            nn.Linear(input_dim, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout_rate)
        )
        self.pos_encoder = PositionalEncoding(d_model=d_model, max_len=seq_len)

        self.mamba_layers = nn.ModuleList()
        self.mamba_norms = nn.ModuleList()
        for _ in range(num_mamba_layers):
            self.mamba_layers.append(Mamba(d_model=d_model, d_state=d_state, d_conv=4, expand=2))
            self.mamba_norms.append(nn.LayerNorm(d_model))

        self.dropout = nn.Dropout(dropout_rate)

        self.trajectory_head = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(d_model // 2, pred_days)
        )

    def forward(self, x):
        x = self.embedding(x)
        x = self.pos_encoder(x)
        for mamba, norm in zip(self.mamba_layers, self.mamba_norms):
            x = x + self.dropout(mamba(norm(x)))

        x_temporal = x[:, -1, :]
        return self.trajectory_head(x_temporal)


# ==========================================
# ⚙️ 2. 訓練設定與模型載入
# ==========================================
FINAL_EPOCHS = 5 # 你的幸運數字
ACCUMULATION_STEPS = 16

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

best_surgery_path = '/content/drive/MyDrive/MarketMamba_V4/Models/V4_Surgery_Best.pth'
final_model_path = '/content/drive/MyDrive/MarketMamba_V4/Models/V4_GodMode_Production.pth'

# 初始化並載入昨天練好的完美大腦
final_surgery_model = MarketMambaV4_Surgery(
    input_dim=len(feature_cols), seq_len=120, d_model=512,
    pred_days=30, num_mamba_layers=8, d_state=64, dropout_rate=0.4
).to(device)

final_surgery_model.load_state_dict(torch.load(best_surgery_path, map_location=device))
print("✅ 成功載入已修復的 Mamba 權重！")

# 🛡️ 解凍所有參數，設定「差異化學習率」
for param in final_surgery_model.parameters():
    param.requires_grad = True

optimizer = optim.AdamW([
    {'params': final_surgery_model.embedding.parameters(), 'lr': 1e-5},
    {'params': final_surgery_model.pos_encoder.parameters(), 'lr': 1e-5},
    {'params': final_surgery_model.mamba_layers.parameters(), 'lr': 1e-5},
    {'params': final_surgery_model.mamba_norms.parameters(), 'lr': 1e-5},
    {'params': final_surgery_model.trajectory_head.parameters(), 'lr': 1e-3}
], weight_decay=1e-4)

criterion = QuantTrajectoryLoss().to(device)

# ==========================================
# 🚀 3. 執行全數據訓練迴圈
# ==========================================
print(f"🚀 啟動全數據煉丹，共計 {FINAL_EPOCHS} 圈...")
for epoch in range(FINAL_EPOCHS):
    final_surgery_model.train()
    train_loss = 0.0
    optimizer.zero_grad()

    train_pbar = tqdm(train_loader, desc=f"Final Epoch {epoch+1:02d}/{FINAL_EPOCHS} [Train]", leave=False)
    for i, (X_batch, Y_batch) in enumerate(train_pbar):
        X, Y = X_batch.squeeze(0).to(device), Y_batch.squeeze(0).to(device)

        with torch.autocast(device_type='cuda', dtype=torch.float16):
            preds = final_surgery_model(X)
            loss = criterion(preds, Y)
            loss = loss / ACCUMULATION_STEPS

        loss.backward()

        if (i + 1) % ACCUMULATION_STEPS == 0 or (i + 1) == len(train_loader):
            torch.nn.utils.clip_grad_norm_(final_surgery_model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()

        train_loss += loss.item() * ACCUMULATION_STEPS

    avg_train_loss = train_loss / len(train_loader)
    print(f"📈 Final Epoch {epoch+1:02d} | Train Loss: {avg_train_loss:.5f}")

# 存下最終上線權重
torch.save(final_surgery_model.state_dict(), final_model_path)
print(f"🎉 實盤大腦進化完成！已存檔至: {final_model_path}")

🏆 啟動實盤前最終煉丹 (吸收 2025-2026 最新記憶)...
✅ 成功載入已修復的 Mamba 權重！
🚀 啟動全數據煉丹，共計 5 圈...


Final Epoch 01/5 [Train]:   0%|          | 0/1535 [00:00<?, ?it/s]

📈 Final Epoch 01 | Train Loss: 0.00730


Final Epoch 02/5 [Train]:   0%|          | 0/1535 [00:00<?, ?it/s]

📈 Final Epoch 02 | Train Loss: 0.00725


...

📈 Final Epoch 03 | Train Loss: 0.00727


Final Epoch 04/5 [Train]:   0%|          | 0/1535 [00:00<?, ?it/s]

📈 Final Epoch 04 | Train Loss: 0.00724


Final Epoch 05/5 [Train]:   0%|          | 0/1535 [00:00<?, ?it/s]

📈 Final Epoch 05 | Train Loss: 0.00726
🎉 實盤大腦進化完成！已存檔至: /content/drive/MyDrive/MarketMamba_V4/Models/V4_GodMode_Production.pth


In [6]:
# ==========================================
# 🦁 最終實盤微創手術 (勇敢者版 - 解除恐懼封印)
# ==========================================
import torch
import torch.nn as nn
import torch.optim as optim
import math
from tqdm.auto import tqdm
import os
from mamba_ssm import Mamba

print("🦁 啟動勇敢者煉丹模式，解除方向懲罰封印...")

# ==========================================
# 🧩 1. 藍圖定義 (確保獨立運行)
# ==========================================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
    def forward(self, x): return x + self.pe[:, :x.size(1), :]

class MarketMambaV4_Surgery(nn.Module):
    def __init__(self, input_dim, seq_len=120, d_model=512, pred_days=30, num_mamba_layers=8, d_state=64, dropout_rate=0.4):
        super().__init__()
        self.embedding = nn.Sequential(nn.Linear(input_dim, d_model), nn.LayerNorm(d_model), nn.GELU(), nn.Dropout(dropout_rate))
        self.pos_encoder = PositionalEncoding(d_model=d_model, max_len=seq_len)
        self.mamba_layers = nn.ModuleList([Mamba(d_model=d_model, d_state=d_state, d_conv=4, expand=2) for _ in range(num_mamba_layers)])
        self.mamba_norms = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(num_mamba_layers)])
        self.dropout = nn.Dropout(dropout_rate)
        # 新的輸出頭
        self.trajectory_head = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(d_model // 2, pred_days)
        )
    def forward(self, x):
        x = self.embedding(x)
        x = self.pos_encoder(x)
        for mamba, norm in zip(self.mamba_layers, self.mamba_norms):
            x = x + self.dropout(mamba(norm(x)))
        return self.trajectory_head(x[:, -1, :])

# ==========================================
# ⚙️ 2. 拔除毒瘤 Loss，換上純粹的 MSE
# ==========================================
class BraveTrajectoryLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.mse = nn.MSELoss(reduction='none') # 最純粹的均方誤差

    def forward(self, pred, target):
        # 排除 target 為 0 的無效填充資料
        valid_mask = torch.where(target == 0.0, 0.0, 1.0)
        loss = self.mse(pred, target) * valid_mask
        return loss.sum() / (valid_mask.sum() + 1e-8)

# ==========================================
# 🚀 3. 載入原始巨獸，進行純粹的 Head 訓練
# ==========================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
FINAL_EPOCHS = 10  # 放開來讓它跑 10 圈
ACCUMULATION_STEPS = 16

# 🚨 注意：我們重新抓取那顆有健康 Mamba 但被 Attention 害死的原始大腦
raw_leviathan_path = '/content/drive/MyDrive/MarketMamba_V4/Models/V4_GodMode_Leviathan_Best.pth'
final_model_path = '/content/drive/MyDrive/MarketMamba_V4/Models/V4_GodMode_Production.pth'

# 借用舊大腦的殼來讀取權重
class OldGodMode(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Sequential(nn.Linear(len(feature_cols), 512), nn.LayerNorm(512), nn.GELU(), nn.Dropout(0.4))
        self.mamba_layers = nn.ModuleList([Mamba(d_model=512, d_state=64, d_conv=4, expand=2) for _ in range(8)])
        self.mamba_norms = nn.ModuleList([nn.LayerNorm(512) for _ in range(8)])
        # 其他省略，我們只要上面這三塊健康的肉

old_state = torch.load(raw_leviathan_path, map_location=device)
final_model = MarketMambaV4_Surgery(input_dim=len(feature_cols)).to(device)

# 移植健康的器官
final_model.embedding.load_state_dict({k.replace('embedding.', ''): v for k, v in old_state.items() if 'embedding.' in k})
final_model.mamba_layers.load_state_dict({k.replace('mamba_layers.', ''): v for k, v in old_state.items() if 'mamba_layers.' in k})
final_model.mamba_norms.load_state_dict({k.replace('mamba_norms.', ''): v for k, v in old_state.items() if 'mamba_norms.' in k})

# ❄️ 絕對凍結 Mamba！我們只准新的 Output Head 去適應健康的特徵
for name, param in final_model.named_parameters():
    if 'trajectory_head' not in name:
        param.requires_grad = False

optimizer = optim.AdamW(final_model.trajectory_head.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = BraveTrajectoryLoss().to(device)

print(f"🚀 啟動勇敢者全數據煉丹，共計 {FINAL_EPOCHS} 圈...")
for epoch in range(FINAL_EPOCHS):
    final_model.train()
    train_loss = 0.0
    optimizer.zero_grad()

    train_pbar = tqdm(train_loader, desc=f"Brave Epoch {epoch+1:02d}/{FINAL_EPOCHS}", leave=False)
    for i, (X_batch, Y_batch) in enumerate(train_pbar):
        X, Y = X_batch.squeeze(0).to(device), Y_batch.squeeze(0).to(device)

        with torch.autocast(device_type='cuda', dtype=torch.float16):
            preds = final_model(X)
            loss = criterion(preds, Y)
            loss = loss / ACCUMULATION_STEPS

        loss.backward()

        if (i + 1) % ACCUMULATION_STEPS == 0 or (i + 1) == len(train_loader):
            optimizer.step()
            optimizer.zero_grad()

        train_loss += loss.item() * ACCUMULATION_STEPS

    avg_train_loss = train_loss / len(train_loader)
    print(f"📈 Epoch {epoch+1:02d} | Train Loss (MSE): {avg_train_loss:.6f}")

torch.save(final_model.state_dict(), final_model_path)
print(f"🎉 勇敢者大腦進化完成！已覆蓋存檔至: {final_model_path}")

🦁 啟動勇敢者煉丹模式，解除方向懲罰封印...
🚀 啟動勇敢者全數據煉丹，共計 10 圈...


Brave Epoch 01/10:   0%|          | 0/1535 [00:00<?, ?it/s]

📈 Epoch 01 | Train Loss (MSE): 0.223868


Brave Epoch 02/10:   0%|          | 0/1535 [00:00<?, ?it/s]

📈 Epoch 02 | Train Loss (MSE): 0.017497


...

📈 Epoch 08 | Train Loss (MSE): 0.016952


Brave Epoch 09/10:   0%|          | 0/1535 [00:00<?, ?it/s]

📈 Epoch 09 | Train Loss (MSE): 0.016953


Brave Epoch 10/10:   0%|          | 0/1535 [00:00<?, ?it/s]

📈 Epoch 10 | Train Loss (MSE): 0.016952
🎉 勇敢者大腦進化完成！已覆蓋存檔至: /content/drive/MyDrive/MarketMamba_V4/Models/V4_GodMode_Production.pth
